In [16]:
%load_ext autoreload
%autoreload 2

from zrp import ZRP
import pandas as pd
import numpy as np

mapping = {
    "FIRST_NAME": "first_name", 
    "MIDDLE_NAME": "middle_name", 
    "LAST_NAME": "last_name", 
    "HOUSE_NUMBER": "house_number", 
    "STREET_ADDRESS": "street_address", 
    "CITY": "city", 
    "STATE": "state", 
    "ZIP_CODE": "zip_code",
    "RECID": "ZEST_KEY",
}

df = pd.read_parquet("data/temp.parquet")

df["ADDRESS"] = (
    df["ADDRESS"]
    .astype(str)
    .str.replace(r"[^\x00-\x7F]+", "", regex=True)
)

df["address_split"] = df["ADDRESS"].str.split(" ")

df["HOUSE_NUMBER"] = (
    df["address_split"]
    .str[0]
    .str.strip()
)

# keep only numeric house numbers
df["HOUSE_NUMBER"] = (
    pd.to_numeric(df["HOUSE_NUMBER"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

df["STREET_ADDRESS"] = np.where(
    df["HOUSE_NUMBER"].isna(),
    df["ADDRESS"],
    df["address_split"].str[1:].str.join(" ")
)

df["ZIP_CODE"] = df["ZIP_CODE"].astype("string")
df["RECID"] = df["RECID"].astype("string")

for c in ("CITY", "STATE", "ZIP_CODE", "HOUSE_NUMBER", "STREET_ADDRESS"):
    df[c] = df[c].replace("", pd.NA)
    
df = df.rename(columns=mapping)

df = df[list(mapping.values())]

mask = ~(
    df["house_number"].isna()
    & df["city"].isna()
    & df["state"].isna()
    & (
        df["street_address"].isna()
        | (df["street_address"] == "CONFIDENTIAL")
    )
)

df = df.loc[mask].reset_index(drop=True)

df.dtypes


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


first_name                object
middle_name               object
last_name                 object
house_number      string[python]
street_address            object
city                      object
state                     object
zip_code          string[python]
ZEST_KEY          string[python]
dtype: object

In [29]:
from pathlib import Path
folder = Path('./artifacts')
for file in folder.iterdir():
	file.unlink()

zest_race_predictor = ZRP()
zest_race_predictor.fit()
zrp_output = zest_race_predictor.transform(df)
zrp_output

Directory already exists
####################################
Processing rows: 0:25000
####################################
Data is loaded
   [Start] Validating input data
     Number of observations: 10000
     Is key unique: True
Directory already exists
   [Completed] Validating input data

   Formatting P1


  0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1230 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1780 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 2430 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 3180 tasks      | elapsed:    0.1s


   Formatting P2
   reduce whitespace

[Start] Preparing geo data

  The following states are included in the data: ['NC']
   ... on state: NC

   Data is loaded
   [Start] Processing geo data
      ...address cleaning


[Parallel(n_jobs=-1)]: Done 4030 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 4980 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 6030 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 7180 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 8430 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 9780 tasks      | elapsed:    0.3s
100%|██████████| 10000/10000 [00:00<00:00, 35992.60it/s]
[Parallel(n_jobs=-1)]: Done 10000 out of 10000 | elapsed:    0.3s finished


      ...replicating address
         ...Base
         ...Number processing...
         House number dataframe expansion is complete! (n=10014)
         ...Base
         ...Map street suffixes...
         ...Mapped & split by street suffixes...
         ...Number processing...

         Address dataframe expansion is complete! (n=13163)
      ...formatting
   [Completed] Processing geo data
   [Start] Mapping geo data
      ...merge user input & lookup table
      ...mapping


...

   [Completed] Validating input geo data
Directory already exists
...Output saved
   [Completed] Mapping geo data

[Completed] Preparing geo data

[Start] Preparing ACS data
   [Start] Validating ACS input data
     Number of observations: 10000
     Is key unique: True

   [Completed] Validating ACS input data

   ...loading ACS lookup tables


   ... combining ACS & user input data
 ...Copy dataframes
 ...Block group
 ...Census tract
 ...Zip code
 ...No match
 ...Merge
 ...Merging complete
[Complete] Preparing ACS data

   [Start] Validating pipeline input data
     Number of observations: 34021
     Is key unique: False
   [Completed] Validating pipeline input data



100%|██████████| 1/1 [00:00<00:00, 2816.86it/s]
[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished
100%|██████████| 1/1 [00:00<00:00, 3423.92it/s]
[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished
100%|██████████| 1/1 [00:00<00:00, 2659.67it/s]
[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished


   ...Proxies generated
...Output saved
...Output saved


,first_name,middle_name,last_name,house_number,street_address,city,state,zip_code,ZEST_KEY,AAPI,AIAN,BLACK,HISPANIC,WHITE,race_proxy,source_zrp_block_group,source_zrp_census_tract,source_zrp_zip_code
0,CHRISTINA,CASTAGNA,AARON,421,WHITT AVE,BURLINGTON,NC,27215,4,0.010164,0.000815,0.010107,0.010211,0.968703,WHITE,1.0,0.0,0.0
1,GENA,HOLT,AARONSON,107,TERRYWOOD CT,HAW RIVER,NC,27258,12,0.000425,0.005697,0.001222,0.003668,0.988988,WHITE,1.0,0.0,0.0
2,ANA-LINN,CHRISTINA,AASEN,1286,PYRTLE DR,GRAHAM,NC,27253,15,0.002188,0.001427,0.009616,0.612746,0.374024,HISPANIC,1.0,0.0,0.0
3,JACK,EDWARD,ABADIE,612,SIDEVIEW ST,GRAHAM,NC,27253,17,0.016160,0.000674,0.008744,0.114376,0.860045,WHITE,1.0,0.0,0.0
4,MYRA,LYNN,ABADIE,612,SIDEVIEW ST,GRAHAM,NC,27253,18,0.019254,0.000824,0.013051,0.182618,0.784253,WHITE,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,BRIAN,KEITH,POFF,4640,PAGETOWN RD #5,BURLINGTON,NC,27217,96133,0.000399,0.013262,0.001050,0.003354,0.981936,WHITE,1.0,0.0,0.0
9996,DAVID,JAMES,POGLITSCH,1618,GRACE LANDING DR,MEBANE,NC,27302,96136,0.001065,0.000373,0.003263,0.013453,0.981847,WHITE,1.0,0.0,0.0
9997,KEVIN,ANDREW,POGLITSCH,1803,MEADOW GREEN DR,GRAHAM,NC,27253,96138,0.051487,0.005903,0.030949,0.088708,0.822953,WHITE,0.0,0.0,1.0
9998,DAKARI,NATHAN SKYLER,POLINGER,1610,ELDER WAY,BURLINGTON,NC,27215,96176,0.002674,0.001786,0.332557,0.016247,0.646735,WHITE,1.0,0.0,0.0


In [3]:
df.to_csv("py311_race_results.csv")

In [46]:
bisg = pd.read_feather('artifacts/bisg_proxy_output.feather')
boost = pd.read_feather('artifacts/proxy_output.feather')

df = bisg.merge(
    boost[["ZEST_KEY", "race_proxy"]],
    on="ZEST_KEY",
    suffixes=("_bisg", "_boost")
)

pair_counts = (
    df.groupby(["race_proxy_bisg", "race_proxy_boost"])
      .size()
      .reset_index(name="count")
)

disagreements = pair_counts[
    pair_counts["race_proxy_bisg"] != pair_counts["race_proxy_boost"]
].sort_values("count", ascending=False)

disagreements


,race_proxy_bisg,race_proxy_boost,count
14,WHITE,BLACK,810
7,BLACK,WHITE,640
15,WHITE,HISPANIC,58
12,WHITE,AAPI,33
11,HISPANIC,WHITE,26
3,BLACK,AAPI,7
6,BLACK,HISPANIC,7
13,WHITE,AIAN,6
1,AAPI,WHITE,5
4,BLACK,AIAN,2


In [50]:
new = boost['race_proxy'].compare(bisg['race_proxy'])
new.dropna()

,self,other
0,BLACK,WHITE
1,BLACK,WHITE
8,WHITE,HISPANIC
11,WHITE,BLACK
22,BLACK,WHITE
...,...,...
9981,WHITE,BLACK
9985,WHITE,BLACK
9988,WHITE,HISPANIC
9991,BLACK,WHITE


In [17]:
voter_df = pd.read_csv('voter_data_robeson.txt', sep='	', encoding='latin1')

# remove rows with missing street addresses
voter_df = voter_df[voter_df['res_street_address'] != 'REMOVED']

# pull relevant columns 
cleaned_df = voter_df[['first_name', 'middle_name', 'last_name', 'res_street_address', 'res_city_desc','state_cd','zip_code', 'race_code', 'ethnic_code']]

# split address
cleaned_df['house_number'] = cleaned_df['res_street_address'].apply(lambda x: x.split()[0])
cleaned_df['street_address'] = cleaned_df['res_street_address'].apply(lambda x: x.split()[1:])
cleaned_df['street_address'] = cleaned_df['street_address'].apply(lambda x: " ".join(x))

# drop old address column
cleaned_df = cleaned_df.drop(columns=['res_street_address'])

# drop na
cleaned_df = cleaned_df.dropna()

# convert zip code to a str
cleaned_df['zip_code'] = cleaned_df['zip_code'].astype(str)

# rename columns
cleaned_df.rename(columns={
	'res_city_desc':'city', 
	'state_cd':'state'
	}, inplace=True)

# add zest key
cleaned_df['ZEST_KEY'] = range(1, len(cleaned_df) + 1)
cleaned_df['ZEST_KEY'] = cleaned_df['ZEST_KEY'].astype(str)

input_df = cleaned_df.drop(columns=['race_code', 'ethnic_code'])

input_df.dtypes



first_name        object
middle_name       object
last_name         object
city              object
state             object
zip_code          object
house_number      object
street_address    object
ZEST_KEY          object
dtype: object

In [18]:
input_df = input_df.iloc[:10000]
input_df.shape

(10000, 9)

In [19]:
from pathlib import Path
folder = Path('./artifacts')
for file in folder.iterdir():
	file.unlink()

zest_race_predictor = ZRP()
zest_race_predictor.fit()
zrp_output = zest_race_predictor.transform(input_df)
zrp_output

Directory already exists
####################################
Processing rows: 0:25000
####################################
Data is loaded
   [Start] Validating input data
     Number of observations: 10000
     Is key unique: True
Directory already exists
   [Completed] Validating input data

   Formatting P1
   Formatting P2
   reduce whitespace

[Start] Preparing geo data

  The following states are included in the data: ['NC']


  0%|          | 0/1 [00:00<?, ?it/s]

   ... on state: NC

   Data is loaded
   [Start] Processing geo data
      ...address cleaning


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1230 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1780 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 2430 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 3180 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 4030 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 4980 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 6030 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 7180 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 8430 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 9780 tasks      | elapsed:    0.1s
100%|██████████| 10000/10000 [00:00<00:0

      ...replicating address
         ...Base
         ...Number processing...


...

   [Completed] Validating input geo data
Directory already exists
...Output saved
   [Completed] Mapping geo data

[Completed] Preparing geo data

[Start] Preparing ACS data
   [Start] Validating ACS input data
     Number of observations: 10000
     Is key unique: True

   [Completed] Validating ACS input data

   ...loading ACS lookup tables


   ... combining ACS & user input data
 ...Copy dataframes
 ...Block group
 ...Census tract
 ...Zip code
 ...No match
 ...Merge
 ...Merging complete
[Complete] Preparing ACS data

   [Start] Validating pipeline input data
     Number of observations: 33985
     Is key unique: False
   [Completed] Validating pipeline input data



100%|██████████| 1/1 [00:00<00:00, 2949.58it/s]


   ...Proxies generated
...Output saved
...Output saved


,first_name,middle_name,last_name,city,state,zip_code,house_number,street_address,ZEST_KEY,AAPI,AIAN,BLACK,HISPANIC,WHITE,race_proxy,source_zrp_block_group,source_zrp_census_tract,source_zrp_zip_code,source_bisg,source_zrp_name_only
0,AJA,VONEE,AARON,LUMBERTON,NC,28358.0,3381,E ELIZABETHTOWN RD #33,1,0.001107,0.001584,0.917611,0.002725,0.076974,BLACK,1.0,0.0,0.0,0.0,0.0
1,BRADLEY,JACOB,ABBOTT,PEMBROKE,NC,28372.0,300,BLUEBILL DR #203,2,0.004908,0.297279,0.018640,0.043770,0.635403,WHITE,0.0,0.0,1.0,0.0,0.0
2,CARLOS,DUAINE,ABBOTT,LUMBERTON,NC,28358.0,67,DORIS E LN,3,0.002039,0.014096,0.023802,0.020132,0.939932,WHITE,0.0,0.0,1.0,0.0,0.0
3,JENNIFER,ROSE GORE,ABBOTT,LUMBERTON,NC,28360.0,3059,WESTMINSTER RD,4,0.066494,0.246732,0.042449,0.093603,0.550722,WHITE,0.0,0.0,1.0,0.0,0.0
4,JUSTIN,GRADY,ABBOTT,LUMBERTON,NC,28360.0,3059,WESTMINSTER RD,5,0.014957,0.305056,0.037033,0.010214,0.632739,WHITE,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,ALFONZO,LUIS,CHACON,RED SPRINGS,NC,28377.0,5,ROBINWOOD DR #C,9996,0.000122,0.002480,0.001984,0.988514,0.006901,HISPANIC,0.0,0.0,1.0,0.0,0.0
9996,CAMILA,GALLARDO,CHACON,LUMBERTON,NC,28358.0,27,TRINITY DR,9997,0.001330,0.000417,0.000535,0.974935,0.022783,HISPANIC,1.0,0.0,0.0,0.0,0.0
9997,LINDA,JOHNSON,CHACON,RED SPRINGS,NC,28377.0,6,ROBINWOOD APTS #C,9998,0.002791,0.212541,0.017646,0.191337,0.575685,WHITE,0.0,0.0,1.0,0.0,0.0
9998,SHAWN,PIERRE,CHACON,SHANNON,NC,28386.0,14571,NC71 HWY N,9999,0.001155,0.170332,0.014246,0.731654,0.082613,HISPANIC,0.0,0.0,1.0,0.0,0.0
